# Cleanse, transform, and load data into Unity Catalog

## Real estate demo scenario

This demo uses data from **PropertyPro NL**, a fictional Dutch real estate agency operating in five major cities: Amsterdam, Rotterdam, Utrecht, The Hague, and Eindhoven.

The agency consolidates property listings from multiple regional offices into Unity Catalog. The raw data contains common real-world quality issues: duplicate records from repeated imports and missing values in optional property attributes.

| Table | Description |
|---|---|
| `properties` | 500 property listings — includes 30 duplicate records and ~56 null values |
| `agents` | 20 real estate agents across five regions |
| `property_updates` | Staging table: 50 price/status updates + 20 new listings (used for MERGE demo) |

Run the cell below to set up the demo schema and tables.


In [ ]:
from scripts.setup import setup, setup_08

spark.sql("USE CATALOG trainer_demo")
setup_08(spark)

## Profile data

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.profile-data-summary-statistics.png)

### 🎯 DEMO: Profile data — summary statistics

**Talking points**
- Before cleansing any data, understand what you have.
- `ANALYZE TABLE` collects row counts, column min/max, null counts, and distinct counts that feed the query optimizer.
- `DESCRIBE TABLE EXTENDED` and `df.summary()` expose those statistics at different levels of detail.

**Demo flow**
1. Analyze the `properties` table to collect statistics.
2. Inspect column-level stats with `DESC EXTENDED`.
3. Use the DataFrame API to generate a quick numeric summary.


In [ ]:
%sql
-- Switch to the demo schema
USE trainer_demo.demo_08;

-- Step 1: Collect statistics for the full table
ANALYZE TABLE properties COMPUTE STATISTICS;

-- Step 2: Collect column-level statistics for key columns
ANALYZE TABLE properties COMPUTE STATISTICS FOR COLUMNS
    listing_price, bedrooms, bathrooms, area_sqm, year_built;

-- Step 3: Inspect collected statistics for listing_price
DESC EXTENDED properties listing_price;


In [ ]:

# Step 4: Use the DataFrame API to profile numeric columns
df = spark.table("trainer_demo.demo_08.properties")

# describe() returns count, mean, std, min, max for numeric columns
df.describe("listing_price", "bedrooms", "bathrooms", "area_sqm", "year_built").show()


## Choose column data types

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.choose-appropriate-column-data-types.png)

### 🎯 DEMO: Choose column data types

**Talking points**
- Type choice affects precision, storage, and query correctness.
- `listing_price` is stored as `DOUBLE` (floating-point). Floating-point arithmetic can introduce tiny rounding errors — unsuitable for financial calculations.
- `DECIMAL(12,2)` stores exact values: up to 10 digits before the decimal and exactly 2 after.
- `year_built` is stored as `INT` (4 bytes). A `SMALLINT` (2 bytes, range −32 768 to 32 767) is sufficient for years.
- Use `CAST` to convert values without rebuilding the table.

**Demo flow**
1. Look at the current schema — note `DOUBLE` for price and `INT` for year_built.
2. Show a precision problem with `DOUBLE` arithmetic.
3. Create a typed view that casts columns to their correct types.


In [ ]:
%sql
USE trainer_demo.demo_08;

-- Step 1: Inspect the current schema
DESCRIBE TABLE properties;

In [ ]:
%sql
-- ✅ Provided: run this to observe floating-point imprecision with DOUBLE
SELECT
  -- Does 0.1 + 0.2 = 0.3 with DOUBLE?
  CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) as sum_double,
  CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) = CAST(0.3 AS DOUBLE)        AS double_exact,   -- returns FALSE
  CAST(0.1 AS DECIMAL(2,1)) + CAST(0.2 AS DECIMAL(2,1)) as sum_decimal,
  CAST(0.1 AS DECIMAL(2,1)) + CAST(0.2 AS DECIMAL(2,1)) = CAST(0.3 AS DECIMAL(2,1))        AS decimal_exact  -- returns TRUE

In [ ]:
%sql
-- Step 3: Create a properly typed view of the properties table
CREATE OR REPLACE VIEW properties_typed AS
SELECT
    property_id,
    address,
    neighborhood,
    city,
    property_type,
    bedrooms,
    CAST(bathrooms    AS DECIMAL(3,1)) AS bathrooms,
    CAST(area_sqm     AS DECIMAL(8,2)) AS area_sqm,
    CAST(listing_price AS DECIMAL(12,2)) AS listing_price,
    listing_date,
    CAST(year_built   AS SMALLINT)     AS year_built,
    status,
    agent_id
FROM properties;

SELECT * FROM properties_typed LIMIT 5;

## Resolve duplicates and nulls

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.resolve-duplicate-missing-null-values.png)

### 🎯 DEMO: Resolve duplicates and nulls

**Talking points**
- The `properties` table was loaded from multiple regional office exports, resulting in 30 duplicate property records.
- Several optional attributes (`bedrooms`, `bathrooms`, `area_sqm`) contain null values where information was not captured.
- Use `GROUP BY / HAVING` or `QUALIFY` with window functions to identify duplicates.
- Use `ROW_NUMBER()` with `QUALIFY` to keep only the most-recently listed version of each property.
- Use `dropna()` / `fillna()` in PySpark or `IS NULL` / `COALESCE` in SQL to handle nulls.

**Demo flow**
1. Count total rows vs distinct property_ids to spot duplicates.
2. Use `QUALIFY` to surface the duplicate rows.
3. Deduplicate using `ROW_NUMBER()`.
4. Count and handle null values in optional columns.


In [ ]:
%sql
USE trainer_demo.demo_08;

-- Step 1: Compare total rows vs distinct property_ids
SELECT
    COUNT(*)                    AS total_rows,
    COUNT(DISTINCT property_id) AS unique_properties,
    COUNT(*) - COUNT(DISTINCT property_id) AS duplicate_rows
FROM properties;

In [ ]:
%sql
-- Step 2: Show duplicate rows using QUALIFY + window function
SELECT *
FROM properties
QUALIFY COUNT(*) OVER (PARTITION BY property_id) > 1
ORDER BY property_id;

In [ ]:
%sql
-- Step 3: Deduplicate — keep the record with the most recent listing_date per property_id
CREATE OR REPLACE TABLE properties_clean AS
SELECT * EXCEPT(rn)
FROM (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY property_id ORDER BY listing_date DESC) AS rn
    FROM properties
)
WHERE rn = 1;

In [ ]:
%sql
SELECT COUNT(*) AS clean_row_count FROM properties_clean;

In [ ]:
%sql
USE trainer_demo.demo_08;

-- Step 4a: Count null values per optional column
SELECT
    COUNT(*)                           AS total_rows,
    COUNT(*) - COUNT(bedrooms)         AS null_bedrooms,
    COUNT(*) - COUNT(bathrooms)        AS null_bathrooms,
    COUNT(*) - COUNT(area_sqm)         AS null_area_sqm
FROM properties_clean;

In [ ]:
%sql
-- Step 4b: Replace nulls — use median bedrooms per property type as a fill value
SELECT
    property_id,
    property_type,
    COALESCE(bedrooms,
        CAST(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY bedrooms)
             OVER (PARTITION BY property_type) AS INT)
    ) AS bedrooms_imputed,
    COALESCE(area_sqm, 0.0) AS area_sqm_filled
FROM properties_clean
LIMIT 20;


In [ ]:
from pyspark.sql.functions import col, count, when, median

df = spark.table("trainer_demo.demo_08.properties_clean")

# Step 4c: PySpark — count nulls across all relevant columns
null_counts = df.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in ["bedrooms", "bathrooms", "area_sqm"]
])
display(null_counts)

In [ ]:
# Step 4d: Drop rows where listing_price is null (critical field)
df_valid = df.dropna(subset=["listing_price"])

# Step 4e: Fill optional nulls with sensible defaults
df_filled = df_valid.fillna({
    "bedrooms":  2,
    "bathrooms": 1.0,
    "area_sqm":  0.0
})
display(df_filled.select("property_id", "property_type", "bedrooms", "bathrooms", "area_sqm").limit(10))

## Transform data with filters and aggregations

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.transform-data-filter-group-aggregate.png)

### 🎯 DEMO: Transform data with filters and aggregations

**Talking points**
- Filtering selects only the records relevant to the current analytical question.
- In PySpark use `filter()` / `where()` — they are identical in behaviour.
- Use `&` and `|` for multi-condition filters; wrap each condition in parentheses.
- `groupBy()` + `agg()` calculates summary statistics per category.
- SQL `GROUP BY` with `HAVING` lets you filter on the aggregated result.

**Demo flow**
1. Filter active listings above a price threshold.
2. Add a second condition (filter by property type).
3. Group by neighborhood and compute price statistics.
4. Use `HAVING` to keep only high-volume neighborhoods.


In [ ]:
%sql
USE trainer_demo.demo_08;

-- Step 1: Filter active listings above €400 000
SELECT property_id, address, city, property_type, listing_price, status
FROM properties_clean
WHERE status = 'Active'
  AND listing_price > 400000
ORDER BY listing_price DESC;

In [ ]:
%sql
-- Step 2: Narrow further — active Amsterdam apartments only
SELECT property_id, address, neighborhood, listing_price
FROM properties_clean
WHERE status = 'Active'
  AND city = 'Amsterdam'
  AND property_type = 'Apartment'
ORDER BY neighborhood, listing_price;

In [ ]:
%sql
-- Step 3: Aggregate — average, min, max price per neighborhood
SELECT
    city,
    neighborhood,
    COUNT(*)                                AS listing_count,
    ROUND(AVG(listing_price), 0)            AS avg_price,
    MIN(listing_price)                      AS min_price,
    MAX(listing_price)                      AS max_price
FROM properties_clean
WHERE status IN ('Active', 'Pending')
GROUP BY city, neighborhood
ORDER BY avg_price DESC;

In [ ]:
%sql
-- Step 4: HAVING — neighborhoods with more than 10 active listings
SELECT
    neighborhood,
    COUNT(*) AS active_listings,
    ROUND(AVG(listing_price), 0) AS avg_price
FROM properties_clean
WHERE status = 'Active'
GROUP BY neighborhood
HAVING COUNT(*) > 10
ORDER BY active_listings DESC;

## Transform data with joins and set operators

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.transform-data-join-union-intersect-except.png)

### 🎯 DEMO: Transform data with joins and set operators

**Talking points**
- **INNER JOIN** returns only rows with a match in both tables — use it to enrich property records with agent info.
- **LEFT ANTI JOIN** returns rows from the left table with *no* match in the right table — useful to find properties that have never been sold.
- **UNION ALL** stacks result sets vertically — use it to combine listings from different status categories.
- **INTERSECT** returns only rows present in both result sets.

**Demo flow**
1. INNER JOIN properties to agents — enrich listings with agency and agent name.
2. LEFT ANTI JOIN — find properties that have no `Sold` record.
3. UNION ALL — combine active listings and sold listings into one unified dataset.


In [ ]:
%sql
USE trainer_demo.demo_08;

-- Step 1: INNER JOIN — enrich active listings with agent and agency details
SELECT
    p.property_id,
    p.address,
    p.city,
    p.property_type,
    p.listing_price,
    p.status,
    a.agent_name,
    a.agency
FROM properties_clean  p
INNER JOIN agents       a ON p.agent_id = a.agent_id
WHERE p.status = 'Active'
ORDER BY p.listing_price DESC
LIMIT 20;

In [ ]:
%sql
-- Step 2: LEFT ANTI JOIN — properties with no Sold counterpart (never sold)
SELECT p.property_id, p.address, p.city, p.listing_price, p.listing_date
FROM properties_clean p
LEFT ANTI JOIN (
    SELECT DISTINCT property_id FROM properties_clean WHERE status = 'Sold'
) sold ON p.property_id = sold.property_id;

In [ ]:
%sql
-- Step 3: UNION ALL — combine active and sold listings with a source label
SELECT property_id, address, city, listing_price, 'Active Listing' AS listing_category
FROM properties_clean
WHERE status = 'Active'
UNION ALL
SELECT property_id, address, city, listing_price, 'Completed Sale' AS listing_category
FROM properties_clean
WHERE status = 'Sold'
ORDER BY listing_price DESC
LIMIT 30;

## Transform data with denormalization and pivots

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.transform-data-denormalize-pivot-unpivot.png)

### 🎯 DEMO: Transform data with denormalization and pivots

**Talking points**
- **Denormalization** flattens multiple normalized tables into one wide table, eliminating runtime joins and speeding up BI queries. Apply this in the gold layer.
- **PIVOT** rotates unique values from a column (e.g. property types) into separate columns — ideal for cross-tabular reports.
- **UNPIVOT** reverses that: converts wide columns back into rows — useful when incoming data arrives in wide format but your pipeline expects long format.

**Demo flow**
1. Create a denormalized gold-layer view that joins properties with agents.
2. PIVOT property counts by `property_type` across neighborhoods.
3. UNPIVOT quarterly listing targets back into rows.


In [ ]:
%sql
USE trainer_demo.demo_08;

-- Step 1: Denormalization — create a gold-layer view combining properties + agents
CREATE OR REPLACE VIEW gold_property_listings AS
SELECT
    p.property_id,
    p.address,
    p.neighborhood,
    p.city,
    p.property_type,
    p.bedrooms,
    p.area_sqm,
    p.listing_price,
    p.listing_date,
    p.year_built,
    p.status,
    a.agent_name,
    a.agency,
    a.region            AS agent_region
FROM properties_clean p
INNER JOIN agents      a ON p.agent_id = a.agent_id;

SELECT * FROM gold_property_listings LIMIT 10;

In [ ]:
%sql
-- Step 2: PIVOT — property count by property_type across neighborhoods (top 6 neighborhoods)
SELECT neighborhood, Apartment, House, Condo, Townhouse, Villa
FROM (
    SELECT neighborhood, property_type
    FROM properties_clean
    WHERE neighborhood IN ('Jordaan', 'De Pijp', 'Kralingen', 'Binnenstad', 'Scheveningen', 'Strijp-S')
)
PIVOT (
    COUNT(*)
    FOR property_type IN (
        'Apartment' AS Apartment,
        'House'     AS House,
        'Condo'     AS Condo,
        'Townhouse' AS Townhouse,
        'Villa'     AS Villa
    )
);

In [ ]:
%sql
-- Step 3: UNPIVOT — convert wide quarterly targets back into rows
CREATE OR REPLACE TEMP VIEW neighborhood_targets (neighborhood, q1_target, q2_target, q3_target, q4_target) AS
VALUES
    ('Jordaan',       45, 52, 48, 55),
    ('De Pijp',       38, 41, 39, 44),
    ('Kralingen',     30, 35, 32, 38),
    ('Binnenstad',    25, 28, 27, 31),
    ('Scheveningen',  20, 22, 21, 25),
    ('Strijp-S',      18, 20, 19, 22);

SELECT neighborhood, quarter, target_listings
FROM neighborhood_targets
UNPIVOT (
    target_listings FOR quarter IN (q1_target AS 'Q1', q2_target AS 'Q2', q3_target AS 'Q3', q4_target AS 'Q4')
)
ORDER BY neighborhood, quarter;


## Load data with merge, insert, and append

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.load-data-merge-insert-append.png)

### 🎯 DEMO: Load data with merge, insert, and append

**Talking points**
- **INSERT INTO** appends rows — use for incremental loads of non-overlapping data (new daily listings). Does not check for existing records.
- **INSERT OVERWRITE** replaces the entire table or a specific partition — use for refreshing aggregated summary tables.
- **MERGE** is the upsert pattern: update matching rows, insert new rows. Essential for synchronising change data.
- `WHEN MATCHED` and `WHEN NOT MATCHED` clauses can have additional conditions for fine-grained control.

**Demo flow**
1. INSERT INTO — append three new property listings.
2. INSERT OVERWRITE — refresh a neighborhood price summary.
3. MERGE — apply the `property_updates` staging table to `properties_clean`.


In [ ]:
%sql
USE trainer_demo.demo_08;

-- ── Step 1: INSERT INTO — append three new listings ──────────────────────────
INSERT INTO properties_clean
    (property_id, address, neighborhood, city, property_type,
     bedrooms, bathrooms, area_sqm, listing_price, listing_date, year_built, status, agent_id)
VALUES
    (501, 'Herengracht 88',    'Centrum',    'Amsterdam', 'Condo',     2, 1.5, 85.0,  325000.0, '2025-03-01', 1998, 'Active', 1),
    (502, 'Coolsingel 14',     'Centrum',    'Rotterdam', 'Apartment', 1, 1.0, 58.0,  189000.0, '2025-03-05', 2010, 'Active', 3),
    (503, 'Stratumseind 77',   'Centrum',    'Eindhoven', 'House',     4, 2.0, 162.0, 498000.0, '2025-03-10', 2003, 'Active', 9);

SELECT COUNT(*) AS total_after_insert FROM properties_clean;

In [ ]:
%sql
-- ── Step 2: INSERT OVERWRITE — refresh the neighborhood price summary ─────────
CREATE TABLE IF NOT EXISTS neighborhood_price_summary (
    city            STRING,
    neighborhood    STRING,
    listing_count   BIGINT,
    avg_price       DOUBLE,
    median_price    DOUBLE
);

INSERT OVERWRITE neighborhood_price_summary
SELECT
    city,
    neighborhood,
    COUNT(*)                                            AS listing_count,
    ROUND(AVG(listing_price), 0)                        AS avg_price,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP
          (ORDER BY listing_price), 0)                  AS median_price
FROM properties_clean
WHERE status IN ('Active', 'Pending')
GROUP BY city, neighborhood;

SELECT * FROM neighborhood_price_summary ORDER BY avg_price DESC LIMIT 10;

In [ ]:
%sql
-- ── Step 3: MERGE — apply status and price updates from staging table ─────────
MERGE INTO properties_clean AS target
USING property_updates     AS source
ON target.property_id = source.property_id
WHEN MATCHED THEN
    UPDATE SET
        target.status        = source.new_status,
        target.listing_price = source.new_price
WHEN NOT MATCHED THEN
    INSERT (property_id, status, listing_price, listing_date,
            address, neighborhood, city, property_type,
            bedrooms, bathrooms, area_sqm, year_built, agent_id)
    VALUES (source.property_id, source.new_status, source.new_price, source.update_date,
            'Unknown', 'Unknown', 'Unknown', 'Unknown',
            NULL, NULL, NULL, NULL, NULL);

-- Verify: check a sample of updated records
SELECT p.property_id, p.status, p.listing_price, u.new_status, u.new_price
FROM properties_clean  p
JOIN property_updates  u ON p.property_id = u.property_id
LIMIT 10;
